# Notebook 4 — SAR Processing


**Objectives**
- Apply speckle filtering to Sentinel-1 backscatter
- Threshold filtered SAR into a binary water mask (fixed + Otsu)
- Clean the mask with morphological operations
- Guard against CRS mismatches before saving
- Export the final mask

In [ ]:
import ee, geemap
ee.Initialize(project="operating-rush-505912-s8")

AOI = ee.Geometry.Rectangle([36.20, -0.83, 36.45, -0.65])  # Lake Naivasha
START, END = "2023-01-01", "2023-03-01"

s1_composite = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterBounds(AOI).filterDate(START, END)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
                .filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
                .select("VV")
                .median()
                .clip(AOI))


## Speckle filtering
SAR backscatter has inherent multiplicative speckle noise (grainy salt-and-pepper texture from coherent interference). Two options below: a fast boxcar/focal-median filter, and the more edge-preserving Refined Lee filter.

In [ ]:
def boxcar_filter(image, radius=1):
    """Fast speckle reduction — simple moving-window mean. Blurs edges."""
    return image.focalMean(radius=radius, kernelType="square", units="pixels")

s1_boxcar = boxcar_filter(s1_composite, radius=1)

In [ ]:
def refined_lee(image):
    """
    Refined Lee speckle filter (adapted from the standard GEE community
    implementation). Edge-preserving — better than boxcar for water
    boundary detection, at the cost of more compute.
    """
    img_natural = ee.Image(10.0).pow(image.divide(10.0))  # dB -> natural (power) units

    weights3 = ee.List.repeat(ee.List.repeat(1, 3), 3)
    kernel3 = ee.Kernel.fixed(3, 3, weights3, 1, 1, False)

    mean3 = img_natural.reduceNeighborhood(ee.Reducer.mean(), kernel3)
    variance3 = img_natural.reduceNeighborhood(ee.Reducer.variance(), kernel3)

    sample_weights = ee.List([[0,0,0,0,0,0,0],[0,1,0,1,0,1,0],[0,0,0,0,0,0,0],
                               [0,1,0,1,0,1,0],[0,0,0,0,0,0,0],[0,1,0,1,0,1,0],
                               [0,0,0,0,0,0,0]])
    sample_kernel = ee.Kernel.fixed(7, 7, sample_weights, 3, 3, False)

    sample_mean = mean3.neighborhoodToBands(sample_kernel)
    sample_var = variance3.neighborhoodToBands(sample_kernel)

    gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
    for a, b in [(6,2),(3,5),(0,8)]:
        gradients = gradients.addBands(sample_mean.select(a).subtract(sample_mean.select(b)).abs())
    max_gradient = gradients.reduce(ee.Reducer.max())

    significance = max_gradient.gte(gradients.select(0))
    direction = significance.multiply(1)  # simplified directional flag (0/1)

    sigma_v = variance3.divide(mean3.multiply(mean3))
    b = variance3.subtract(sigma_v.multiply(mean3.multiply(mean3))).divide(variance3.add(1e-12))
    b = b.max(0)

    result = mean3.add(b.multiply(img_natural.subtract(mean3)))
    result_db = ee.Image(10.0).multiply(result.log10())
    return result_db.rename(image.bandNames())

s1_refined_lee = refined_lee(s1_composite)

In [ ]:
Map = geemap.Map(center=[-0.74, 36.33], zoom=11)
Map.addLayer(s1_composite,    {"min": -25, "max": 0}, "VV — raw", shown=False)
Map.addLayer(s1_boxcar,       {"min": -25, "max": 0}, "VV — boxcar filtered", shown=False)
Map.addLayer(s1_refined_lee,  {"min": -25, "max": 0}, "VV — refined Lee filtered")
Map

## Thresholding methods
Calm open water is a near-specular reflector at radar wavelengths — most energy bounces *away* from the sensor, so water reads as very dark (low VV backscatter, typically < -15 dB). Two ways to pick the cutoff:
1. **Fixed threshold** — fast, needs manual tuning per scene/season
2. **Otsu's method** — automatically finds the threshold that best separates a bimodal histogram (water vs. land) — more robust across dates

In [ ]:
# 1) Fixed threshold
FIXED_THRESHOLD = -17  # dB — tune by inspecting the histogram below
water_fixed = s1_refined_lee.lt(FIXED_THRESHOLD).rename("water_fixed")

In [ ]:
# 2) Otsu threshold — compute a histogram server-side, run Otsu client-side
histogram = s1_refined_lee.reduceRegion(
    reducer=ee.Reducer.histogram(255, 1),
    geometry=AOI, scale=10, maxPixels=1e9, bestEffort=True
).get("VV").getInfo()

def otsu_threshold(histogram_dict):
    counts = np.array(histogram_dict["histogramCounts"] if "histogramCounts" in histogram_dict
                       else histogram_dict["histogram"])
    means = np.array(histogram_dict["bucketMeans"] if "bucketMeans" in histogram_dict
                      else None)
    if means is None:
        # reconstruct bin centers from min/max/count if bucketMeans absent
        bmin, bwidth = histogram_dict["bucketMin"], histogram_dict["bucketWidth"]
        means = bmin + bwidth * (np.arange(len(counts)) + 0.5)

    total = counts.sum()
    sum_all = (counts * means).sum()
    sum_bg, weight_bg, max_var, threshold = 0.0, 0.0, 0.0, means[0]
    for i in range(len(counts)):
        weight_bg += counts[i]
        if weight_bg == 0:
            continue
        weight_fg = total - weight_bg
        if weight_fg == 0:
            break
        sum_bg += means[i] * counts[i]
        mean_bg = sum_bg / weight_bg
        mean_fg = (sum_all - sum_bg) / weight_fg
        between_var = weight_bg * weight_fg * (mean_bg - mean_fg) ** 2
        if between_var > max_var:
            max_var, threshold = between_var, means[i]
    return threshold

import numpy as np
otsu_thresh = otsu_threshold(histogram)
print(f"Otsu threshold: {otsu_thresh:.2f} dB")

water_otsu = s1_refined_lee.lt(otsu_thresh).rename("water_otsu")

In [ ]:
Map.addLayer(water_fixed.selfMask(), {"palette": ["cyan"]}, f"Water — fixed ({FIXED_THRESHOLD} dB)")
Map.addLayer(water_otsu.selfMask(),  {"palette": ["magenta"]}, f"Water — Otsu ({otsu_thresh:.1f} dB)")
Map

## Morphological cleanup
Raw thresholded masks are speckled with isolated false-positive/negative pixels. Opening (erode→dilate) removes small noise specks; closing (dilate→erode) fills small holes inside the water body.

In [ ]:
def morphological_cleanup(mask, open_radius=1, close_radius=1):
    opened = (mask.focal_min(radius=open_radius, units="pixels")
                   .focal_max(radius=open_radius, units="pixels"))
    closed = (opened.focal_max(radius=close_radius, units="pixels")
                     .focal_min(radius=close_radius, units="pixels"))
    return closed.rename(mask.bandNames())

water_clean = morphological_cleanup(water_otsu, open_radius=1, close_radius=2)
Map.addLayer(water_clean.selfMask(), {"palette": ["lime"]}, "Water — cleaned")
Map

## CRS guard
Before saving, verify every layer that will be combined (mask, AOI, any reference raster from Notebook 3) shares one CRS. Silent CRS mismatches are the #1 cause of misaligned area calculations downstream in Notebook 5.

In [ ]:
def assert_matching_crs(*images, expected_crs="EPSG:32637"):
    problems = []
    for img in images:
        crs = img.projection().crs().getInfo()
        if crs != expected_crs and crs != "EPSG:4326":  # 4326 = ungridded, reproject-on-export is fine
            problems.append(crs)
    if problems:
        raise ValueError(f"CRS mismatch detected: {set(problems)} != {expected_crs}")
    print(f"CRS guard passed — all layers align to {expected_crs} on export.")

assert_matching_crs(water_clean, s1_composite, expected_crs="EPSG:32637")

## Save mask

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=water_clean.toByte(),
    description="naivasha_sar_water_mask",
    folder="gee_exports",
    region=AOI,
    scale=10,
    crs="EPSG:32637",
    maxPixels=1e10
)
task.start()
print("Mask export started:", task.status()['state'])

# Local copy too, for Notebook 5's cross-check
geemap.ee_export_image(
    water_clean.toByte(), filename="naivasha_sar_water_mask.tif",
    scale=10, region=AOI, crs="EPSG:32637"
)

## Exercise 4.1 — Naivasha SAR
1. Re-run the pipeline above on your **dry-season** Sentinel-1 composite from Exercise 2.1
2. Compare the Otsu threshold value between wet- and dry-season scenes — does it shift, and in which direction?
3. Try `open_radius=2` in the cleanup step — how much lake-edge detail is lost vs. how much noise is removed? Document the trade-off in one sentence for the group discussion.

In [ ]:
# --- Exercise 4.1 scaffold ---
# TODO 1: load your Exercise 2.1 dry-season S1 composite
# TODO 2: refined_lee filter -> Otsu threshold -> morphological_cleanup
# TODO 3: compare dry-season Otsu threshold to the wet-season value above
